# Week 1 Day 2 challenge: a lean agent for UnderCover

Varun Singh

**The challenge:** using the agent, get at least 15 of the 25 claims right on every run, as cheaply as possible, then push for more.

**The result:** on Luna (`openai/gpt-6-luna` at `low`), the lean agent's worst run is 22 of 25 on the queue and 22 on the held-out claims, against 17 and 17 for the lab's agent. It costs about 30% less, runs about a third faster and makes about half as many model calls.

- **Queue:** the lab's 25 claims (`data/claims.json`). I used them to build and tune the agent.
- **Tuning:** running the agent, reading the claims it got wrong, and changing its prompt and tools to fix them. Because the changes come from the queue's mistakes, the queue score can flatter the agent.
- **Held-out set:** 25 other claims with the same mix (`data/evals/held_out_claims.json`), never looked at while tuning. It shows whether the gains carry over to claims the agent hasn't seen.

| Luna `low`, 3 runs each | Queue: worst / mean | Held-out: worst / mean | Cost per run (held-out) | Model calls per claim |
|---|---|---|---|---|
| The lab's agent | 17 / 17.7 | 17 / 17.0 | $0.0070 | 3.0 |
| Lean agent (version 6, short wording) | 22 / 23.3 | 22 / 23.0 | $0.0049 | 1.6 |

**The main takeaway: change the agent before you change the model.** The model stayed the same throughout. The whole gain came from four changes to how the agent works: fixed steps move from tools into code, code does the arithmetic, the agent gets the information it was missing, and it reasons before it decides.

**The idea.** Part 4 of the lab showed that, on the same model, the agent was slower and dearer than a direct call and no more accurate, because its workflow was fixed. So instead of looking for a cheaper model to run the lab's agent, this notebook changes the agent first:

1. **Fixed steps move from tools into code.** The policy record, the plan limits and the customer's earlier claims go in the prompt, and recording the decision ends the run.
2. **Code does the arithmetic.** A payout calculator tool applies section 10 of the wording, and the model decides what's eligible.
3. **The agent gets the information it was missing:** reminders of rules in the wording and a summary of the claims manual.
4. **It reasons before it decides:** `record_decision` asks for the rationale first.

**The method.**

1. Part 1: add one change at a time on the same model, 3 runs each, and judge on the worst run, because the requirement is "every run".
2. Part 2: read the mistakes of the best version, trace each one to its cause, and fix it from the source documents (the manual and the plan data), never from the handlers' answers. That gives version 6.
3. Part 3: run version 6 and the lab's agent on the 25 held-out claims, which were never looked at while tuning.

The outputs are cleared. The results of my runs (2026-11-16 and 2026-11-18) are written into the markdown after each part. Running everything costs about $0.20 on Luna and takes about 20 minutes.

## Setup

In [ ]:
import json
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import date, timedelta
from pathlib import Path
from typing import Literal
import os

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
print("OpenRouter API key found" if openrouter_api_key else "No OPENROUTER_API_KEY in .env")

## The data

The same seed data as the lab. The handlers' decisions (`labels`) are used **only** to score results. No prompt or tool ever reads them.

Depending on the version, the agent can see three things the lab's agent can't: the plan limits, the claims manual, and the customer's earlier claims. From the customer record it gets **only the list of earlier claims**, not their name, address, phone or date of birth.

In [ ]:
DATA = Path("../../data")


def load(name):
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(held_out=False):
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    return claims, {d["claim_id"]: d for d in decisions}


claims, labels = get_sample_data()
held_claims, held_labels = get_sample_data(held_out=True)

policies = {p["policy_id"]: p for p in load("policies.json")}
customers = {c["customer_id"]: c for c in load("customers.json")}
plans = {p["plan"]: p for p in load("plans.json")["plans"]}
plan_settings = load("plans.json")
company = load("company.json")
wording_full = (DATA / "policy_wording.md").read_text(encoding="utf-8")
wording_short = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")
manual = (DATA / "claims_manual.md").read_text(encoding="utf-8")

# The manual split into its numbered sections, for the lookup tool
manual_sections = {}
for block in manual.split("\n## ")[1:]:
    title, _, body = block.partition("\n")
    manual_sections[title.split(". ", 1)[1].strip()] = body.strip()

print(f"{len(claims)} queue claims, {len(held_claims)} held-out claims, {len(policies)} policies")
print(f"Wording: {len(wording_full.split()):,} words full, {len(wording_short.split()):,} short. Manual: {len(manual.split()):,} words")
print("Manual sections:", ", ".join(manual_sections))
print(Counter(label["difficulty"] for label in labels.values()))

## The building blocks

Each version of the agent is described by a small config. Each setting switches one change on or off:

| Setting | Values | What it changes |
|---|---|---|
| `policy` | `"tool"` / `"inject"` | The agent fetches the policy record with a tool (the lab's agent), or code puts it in the prompt |
| `end_on_record` | `False` / `True` | When `True`, the run ends as soon as the decision is recorded, which saves the final "done" call |
| `calculator` | `False` / `True` | Gives the agent a payout calculator |
| `rules` | `False` / `True` | Adds three reminders to the prompt about details of the wording that small models miss |
| `history` | `None` / `"inject"` / `"tool"` | The customer's earlier claims: none, put in the prompt, or a tool the agent can call |
| `manual` | `None` / `"summary"` / `"summary_v2"` / `"tool"` | The claims manual: none, a short summary in the prompt (the first draft, or the completed one from version 6), or a tool that returns one section |
| `wording` | `"full"` / `"short"` | The full policy wording or the short version |
| `plan_limits` | `False` / `True` | Version 6: code adds the customer's plan limits (waiting periods, per-item limit, deductible, maximums) to the prompt |
| `rationale_first` | `False` / `True` | Version 6: `record_decision` asks for the rationale before the decision |

### The prompt

The instructions are the lab's. The other blocks are added according to the config. The system prompt is the same for every claim, so providers that support prompt caching charge less for it after the first claim.

In [ ]:
INSTRUCTIONS = '''You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
'''

# D: details the wording already states, which small models tend to miss
RULES = '''Points to check carefully:
- Trip Delay needs the carrier's confirmation of the delay AND its cause. A carrier document that confirms the delay but does not state the cause is incomplete: treat it as a missing document and refer.
- For belongings the per-item limit applies to each item. A pair or a set (shoes, a suit, a camera with its lens) is one item. A line listing several things, such as "four shirts", is several separate items.
- Electronics Plus raises the limit only for the devices it lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. Other gadgets, such as headphones or game consoles, keep the normal per-item limit.
'''

# B, as a summary: the rules from the claims manual that the wording leaves out
MANUAL_SUMMARY = '''Rules from the claims manual, which bind every decision:
- Documents: theft needs a police report made within 24 hours; items claimed above $100 need proof of value (a receipt or card statement). A missing required document means refer, never decline. If the customer says a required step was never taken (for example the theft was never reported), decline.
- A customer's third claim filed within any 12 months is referred for review, whatever its merits.
- Refer, never decline, when the documents contradict the story, look altered, or the same loss has already been claimed.
- A loss caused by a wild animal (monkey, gull, bear, giraffe and similar) is accidental loss, not theft: no police report is needed, and a written statement from a hotel, tour operator, guide or park authority is enough evidence. Rats and mice are vermin and stay excluded.
- Meals during a covered Trip Delay can be paid without receipts at $35 per traveler for each 24 hours of delay or part of 24 hours. Accommodation, transport and rebooking always need receipts.
- Reasonable accommodation during a delay is up to $350 a night per room, one room for every two travelers.
- Payouts above $1,000 are still approved; payment waits for a handler's sign-off.
'''

# Version 6: the manual summary completed with the counterweights the first summary left out
MANUAL_SUMMARY_V2 = '''Rules from the claims manual, which bind every decision:
- Refer only for these reasons: a required document is missing, the story and the documents disagree, it is the customer's third claim in 12 months, or the wording and these rules do not settle the question. A claim that is clearly covered is approved and a claim that is clearly excluded is declined, even when the amount is large or the story is unusual.
- Documents: theft needs a police report made within 24 hours. Proof of value (a receipt or card statement) is needed only for an individual item claimed above $100. For items of $100 or less, the customer's own list is enough. Judge this item by item: "four shirts, $120" is four items of $30. A missing required document means refer, never decline. If the customer says a required step was never taken (for example the theft was never reported), decline.
- A customer's third claim filed within any 12 months is referred for review, whatever its merits.
- Refer when the documents contradict the story, look altered, or the same loss has already been claimed.
- A loss caused by a wild animal (monkey, gull, bear, giraffe and similar) is accidental loss, not theft: no police report is needed, and a written statement from a hotel, tour operator, guide or park authority is enough evidence. Rats and mice are vermin and stay excluded.
- Meals during a covered Trip Delay: if there are receipts for the meals, pay the receipts. Only when there are no meal receipts, pay a flat $35 per traveler for each 24 hours of delay or part of 24 hours. Meals without receipts are not a missing document. Accommodation, transport and rebooking always need receipts.
- Reasonable accommodation during a delay is up to $350 a night per room, one room for every two travelers.
- Payouts above $1,000 are still approved; payment waits for a handler's sign-off.
'''

# Version 6: with the plan limits injected, check the benefit's conditions against them first
PLAN_CHECK = '''The customer's plan limits are given with the claim. Check the conditions of the benefit against them before anything else. In particular, a Trip Delay or Baggage Delay shorter than the plan's waiting period is not covered and is declined.
'''


def plan_limits_text(claim):
    '''The limits of this customer's plan, from plans.json, so the model does not have to find them in the wording's table.'''
    policy = policies[claim["policy_id"]]
    plan = plans[policy["plan"]]
    lines = [
        f"Plan: {plan['name']}" + (f", with add-ons: {', '.join(policy['add_ons'])}" if policy["add_ons"] else ", no add-ons"),
        f"- Trip Cancellation and Interruption: maximum ${plan['trip_cancellation_max']:,.0f}, or the trip cost of ${policy['trip']['trip_cost']:,.0f} if lower",
        f"- Trip Delay: waiting period {plan['trip_delay_waiting_hours']} hours, maximum ${plan['trip_delay_max']:,.0f}",
        f"- Baggage Delay: waiting period {plan['baggage_delay_waiting_hours']} hours, maximum ${plan['baggage_delay_max']:,.0f}",
        f"- Baggage and Personal Effects: maximum ${plan['baggage_max']:,.0f}, per-item limit ${plan['per_item_limit']:,.0f}"
        + (" ($1,200 for listed electronics, with Electronics Plus)" if "electronics_plus" in policy["add_ons"] else ""),
        f"- Passport and Travel Documents: maximum ${plan['passport_max']:,.0f}",
        f"- Emergency Medical: maximum ${plan['emergency_medical_max']:,.0f}",
        f"- Deductible: ${plan['deductible']:,.0f}, once per claim, on Baggage and Personal Effects and Emergency Medical only",
    ]
    return "\n".join(lines)


TOOLS_LINE = {
    "policy": "Use get_policy to look up the customer's policy record.",
    "calculator": "When you approve, use calculate_payout to work out the payout, then record exactly the amount it returns.",
    "history": "Use get_claim_history to check the customer's earlier claims when it could affect the decision.",
    "manual": "Use get_manual_section to read a section of the claims manual when the wording does not settle a question.",
}


def make_system_prompt(cfg):
    parts = [INSTRUCTIONS]
    if cfg["rules"]:
        parts.append(RULES)
    if cfg["manual"] == "summary":
        parts.append(MANUAL_SUMMARY)
    elif cfg["manual"] == "summary_v2":
        parts.append(MANUAL_SUMMARY_V2)
    if cfg.get("plan_limits"):
        parts.append(PLAN_CHECK)
    tool_lines = ["Record your decision with record_decision once you have decided."]
    if cfg["policy"] == "tool":
        tool_lines.insert(0, TOOLS_LINE["policy"])
    for key in ["calculator", "history", "manual"]:
        if cfg[key] in (True, "tool"):
            tool_lines.insert(-1, TOOLS_LINE[key])
    parts.append("\n".join(tool_lines))
    parts.append(wording_short if cfg["wording"] == "short" else wording_full)
    return "\n\n".join(parts)


def recent_claims(claim):
    '''The customer's earlier claims, and how many claims (including this one) were filed in the 12 months to this one.'''
    previous = customers[claim["customer_id"]].get("previous_claims", [])
    filed = date.fromisoformat(claim["submitted_on"])
    window_start = filed - timedelta(days=365)
    in_window = [p for p in previous if window_start <= date.fromisoformat(p["filed_on"]) <= filed]
    return {"previous_claims": previous, "claims_filed_in_12_months_including_this_one": len(in_window) + 1}


def make_user_prompt(claim, cfg):
    parts = []
    if cfg["policy"] == "inject":
        parts.append("# Policy record\n\n" + json.dumps(policies[claim["policy_id"]], indent=2))
    if cfg.get("plan_limits"):
        parts.append("# Plan limits\n\n" + plan_limits_text(claim))
    if cfg["history"] == "inject":
        parts.append("# Customer's claim history\n\n" + json.dumps(recent_claims(claim), indent=2))
    parts.append("# Claim\n\n" + json.dumps(claim, indent=2))
    return "\n\n".join(parts)

### The tools

- `record_decision` and `get_policy` are the lab's tools. With `end_on_record`, `record_decision` has `return_direct=True`, so LangChain stops the loop once it runs.
- `calculate_payout` follows section 10 of the wording in plain Python. **The model still decides what's eligible**: which lines count, and how to group them into items. The calculator applies the per-item limits, takes off what others paid and the deductible, and caps the result at the benefit's maximum.
- `get_claim_history` returns only the customer's earlier claims, with a count of claims in the last 12 months worked out by code.
- `get_manual_section` returns one section of the claims manual.

The decision is read from the arguments of the `record_decision` call in the agent's messages, so runs can safely go in parallel.

In [ ]:
def make_record_tool(end_run, rationale_first=False):
    '''The lab's record_decision tool. end_run=True stops the agent once it runs.
    rationale_first=True (version 6) puts the rationale before the decision, so the model writes its reasoning before it commits.'''
    if rationale_first:
        @tool("record_decision", return_direct=end_run)
        def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
            '''Record the decision on a claim in the claims system.

            rationale is 2-3 sentences to explain the decision along with the relevant section of the policy. Write it first.
            payout is the amount to pay in US dollars, or 0 for a decline or a referral.
            '''
            return f"Recorded {decision} on {claim_id}"
    else:
        @tool("record_decision", return_direct=end_run)
        def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
            '''Record the decision on a claim in the claims system.

            payout is the amount to pay in US dollars, or 0 for a decline or a referral.
            rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
            '''
            return f"Recorded {decision} on {claim_id}"
    return record_decision


RECORD_TOOLS = {(end, first): make_record_tool(end, first) for end in (False, True) for first in (False, True)}


@tool
def get_policy(policy_id: str) -> str:
    '''Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers.'''
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


class PayoutLine(BaseModel):
    description: str
    amount: float = Field(description="The eligible amount for this one item or expense, in US dollars, before any limit")
    listed_electronics: bool = Field(
        default=False,
        description="True only for a phone, tablet, laptop, camera or its lens, e-reader or smartwatch",
    )


BENEFIT_MAX = {
    "baggage": "baggage_max",
    "baggage_delay": "baggage_delay_max",
    "trip_delay": "trip_delay_max",
    "passport": "passport_max",
    "emergency_medical": "emergency_medical_max",
    "trip_cancellation": "trip_cancellation_max",
    "trip_interruption": "trip_cancellation_max",
}


def payout_calculation(policy_id, benefit, lines, paid_by_others=0.0):
    '''Section 10 of the wording, in code. Returns the payout and the steps.'''
    policy = policies[policy_id]
    plan = plans[policy["plan"]]
    electronics_limit = next(a["electronics_per_item_limit"] for a in plan_settings["add_ons"] if a["add_on"] == "electronics_plus")
    steps, total = [], 0.0
    for line in lines:
        line = line if isinstance(line, PayoutLine) else PayoutLine(**line)
        amount = line.amount
        if benefit == "baggage":
            limit = electronics_limit if (line.listed_electronics and "electronics_plus" in policy["add_ons"]) else plan["per_item_limit"]
            if amount > limit:
                steps.append(f"{line.description}: ${line.amount:,.2f} capped at the per-item limit of ${limit:,.2f}")
                amount = limit
            else:
                steps.append(f"{line.description}: ${amount:,.2f}")
        else:
            steps.append(f"{line.description}: ${amount:,.2f}")
        total += amount
    steps.append(f"Eligible total: ${total:,.2f}")
    if paid_by_others:
        total -= paid_by_others
        steps.append(f"Less paid by the carrier or others: -${paid_by_others:,.2f}")
    if benefit in plan_settings["deductible_applies_to"] and plan["deductible"]:
        total -= plan["deductible"]
        steps.append(f"Less the {policy['plan']} deductible: -${plan['deductible']:,.2f}")
    maximum = plan[BENEFIT_MAX[benefit]]
    if benefit in ("trip_cancellation", "trip_interruption"):
        maximum = min(maximum, policy["trip"]["trip_cost"])
    if total > maximum:
        steps.append(f"Capped at the maximum for the benefit: ${maximum:,.2f}")
        total = maximum
    payout = round(max(total, 0.0), 2)
    return payout, steps


@tool
def calculate_payout(
    policy_id: str,
    benefit: Literal["baggage", "baggage_delay", "trip_delay", "passport", "emergency_medical", "trip_cancellation", "trip_interruption"],
    lines: list[PayoutLine],
    paid_by_others: float = 0.0,
) -> str:
    '''Work out the payout for an approved claim, following section 10 of the policy wording.

    You decide what is eligible; this tool does the arithmetic.
    - lines: one entry per eligible item or expense. Leave out excluded lines.
      For belongings, a pair or a set (shoes, a suit, a camera with its lens) is ONE line with the combined amount,
      and a line such as "four shirts at $85" is FOUR separate lines.
    - paid_by_others: anything a carrier, hotel or other party has already paid for the same loss.
    The tool applies the per-item limit (and Electronics Plus where the policy has it), the deductible and the benefit maximum.
    '''
    payout, steps = payout_calculation(policy_id, benefit, lines, paid_by_others)
    note = " If the payout is 0, the claim is declined." if payout == 0 else ""
    return "\n".join(steps) + f"\nPayout: ${payout:,.2f}.{note}"


def make_history_tool(claim):
    @tool
    def get_claim_history() -> str:
        '''Look up the customer's earlier claims with UnderCover, with the number of claims filed in the 12 months to this one.'''
        return json.dumps(recent_claims(claim), indent=2)
    return get_claim_history


ManualSection = Literal[tuple(manual_sections)]


@tool
def get_manual_section(section: ManualSection) -> str:
    '''Read one section of UnderCover's internal claims manual. It covers what the policy wording leaves open:
    required documents, interpretation notes (such as wild animals, unattended belongings, reasonable accommodation),
    when the story and documents disagree, repeat claims and meals without receipts.'''
    return manual_sections[section]

### Running a version on a set of claims

`decide_claim` runs the agent on one claim and scores it. It also counts the model calls, and any cached input tokens the provider reports. `run_version` runs every claim, a few at a time, and prints one line per run.

A claim is right when the decision matches the handler's and the payout is within a dollar, as in the lab. An error (a timeout, a loop, a rate limit that outlasts the retries) counts as wrong.

In [ ]:
def make_llm(model, effort):
    kwargs = dict(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)
    if effort:
        kwargs["reasoning_effort"] = effort
    return ChatOpenAI(**kwargs)


def make_tools(cfg, claim):
    tools = [RECORD_TOOLS[(cfg["end_on_record"], cfg.get("rationale_first", False))]]
    if cfg["policy"] == "tool":
        tools.append(get_policy)
    if cfg["calculator"]:
        tools.append(calculate_payout)
    if cfg["history"] == "tool":
        tools.append(make_history_tool(claim))
    if cfg["manual"] == "tool":
        tools.append(get_manual_section)
    return tools


def make_agent(llm, cfg, claim):
    return create_agent(model=llm, tools=make_tools(cfg, claim), system_prompt=make_system_prompt(cfg))


def is_right(answer, label):
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def decide_claim(llm, cfg, claim, label):
    start = time.time()
    answer = {"decision": "nothing recorded", "payout": 0.0, "rationale": ""}
    cost, calls, cached, tools_used, error = 0.0, 0, 0, [], ""
    try:
        agent = make_agent(llm, cfg, claim)
        result = agent.invoke(
            {"messages": [{"role": "user", "content": make_user_prompt(claim, cfg)}]},
            config={"recursion_limit": 16},
        )
        for message in result["messages"]:
            if message.type != "ai":
                continue
            calls += 1
            usage = message.response_metadata.get("token_usage") or {}
            cost += usage.get("cost") or 0.0
            cached += (usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0
            for call in message.tool_calls:
                tools_used.append(call["name"])
                if call["name"] == "record_decision":
                    args = call["args"]
                    answer = {"decision": args.get("decision"), "payout": float(args.get("payout") or 0), "rationale": args.get("rationale", "")}
    except Exception as exc:
        error = f"{type(exc).__name__}: {str(exc)[:200]}"
    return {
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "handler": label["decision"],
        "handler_payout": label["payout"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
        "model_calls": calls,
        "cached_tokens": cached,
        "tools_used": ", ".join(tools_used),
        "error": error,
    }


def run_version(name, cfg, model, effort, claims, labels, runs=1, workers=5, dataset="queue"):
    '''Run one version of the agent on one model, `runs` times. Returns one row per claim per run.'''
    llm = make_llm(model, effort)
    rows = []
    for run in range(1, runs + 1):
        start = time.time()
        with ThreadPoolExecutor(max_workers=workers) as pool:
            results = list(pool.map(lambda c: decide_claim(llm, cfg, c, labels[c["claim_id"]]), claims))
        wall = time.time() - start
        for r in results:
            r.update({"version": name, "model": model, "effort": effort or "default", "run": run, "dataset": dataset, "wall_seconds": round(wall, 1)})
        rows += results
        right = sum(r["right"] for r in results)
        errors = sum(bool(r["error"]) for r in results)
        print(f"{name} | {model} {effort or ''} | run {run}: {right}/{len(results)} right, "
              f"${sum(r['cost'] for r in results):.4f}, {wall:.0f}s wall, "
              f"{sum(r['model_calls'] for r in results) / len(results):.1f} calls a claim"
              + (f", {errors} errors" if errors else ""))
    return rows


def summarize(rows, by=("version", "model", "effort")):
    '''One line per version and model: worst, mean and best run, cost and time per run, calls per claim.'''
    df = pd.DataFrame(rows)
    per_run = df.groupby([*by, "run"], sort=False).agg(
        right=("right", "sum"), cost=("cost", "sum"), wall=("wall_seconds", "first"),
        calls=("model_calls", "mean"), errors=("error", lambda e: (e != "").sum()),
    ).reset_index()
    out = per_run.groupby(list(by), sort=False).agg(
        runs=("run", "count"), worst=("right", "min"), mean=("right", "mean"), best=("right", "max"),
        cost_per_run=("cost", "mean"), seconds_per_run=("wall", "mean"),
        calls_per_claim=("calls", "mean"), errors=("errors", "sum"),
    ).reset_index()
    out["mean"] = out["mean"].round(1)
    out["cost_per_run"] = out["cost_per_run"].round(5)
    out["seconds_per_run"] = out["seconds_per_run"].round(0)
    out["calls_per_claim"] = out["calls_per_claim"].round(1)
    return out


def by_difficulty(rows, by=("version", "model")):
    '''Mean number right at each difficulty, across runs.'''
    df = pd.DataFrame(rows)
    runs = df.groupby(list(by), sort=False)["run"].nunique()
    table = df.pivot_table(index=list(by), columns="difficulty", values="right", aggfunc="sum", sort=False)
    table = table.div(runs, axis=0).round(1)
    return table[[c for c in ["routine", "hard", "manual"] if c in table.columns]]


def show_misses(rows, version, model, run=1):
    for r in rows:
        if r["version"] == version and r["model"] == model and r["run"] == run and not r["right"]:
            display(Markdown(
                f"**{r['claim_id']}, {r['difficulty']}**: {r['decision']} ${r['payout']:,.2f}, "
                f"handler said {r['handler']} ${r['handler_payout']:,.2f}. Tools: {r['tools_used'] or 'none'}"
                + (f" **Error:** {r['error']}" if r["error"] else "")
                + f"\n\n> {r['rationale']}"
            ))

### Checking the calculator first

The calculator is plain code, so it can be checked for free against the handlers' worked calculations before any model uses it. These are three of the claims where the arithmetic is the hard part, grouped into items the way the handler did.

In [ ]:
checks = {
    "CLM-1011": dict(benefit="baggage", paid_by_others=350, lines=[
        {"description": "Suitcase", "amount": 260}, {"description": "Tuxedo, jacket and trousers", "amount": 480},
        {"description": "Dress shoes", "amount": 230}, {"description": "Camera body with its lens", "amount": 520, "listed_electronics": True},
        {"description": "Cufflinks", "amount": 120}, *[{"description": "Dress shirt", "amount": 85}] * 4]),
    "CLM-1014": dict(benefit="baggage", paid_by_others=100, lines=[
        {"description": "Duffel bag", "amount": 110}, *[{"description": "Polo shirt", "amount": 65}] * 3,
        *[{"description": "Swimsuit", "amount": 85}] * 2, *[{"description": "Shorts", "amount": 60}] * 2,
        {"description": "Sandals", "amount": 140}, {"description": "Snorkel set", "amount": 70}]),
    "CLM-1021": dict(benefit="baggage", lines=[
        {"description": "Headphones", "amount": 429}, {"description": "E-reader", "amount": 180, "listed_electronics": True},
        {"description": "Game console", "amount": 399}, {"description": "Tote bag", "amount": 95}]),
}

for claim_id, args in checks.items():
    policy_id = next(c["policy_id"] for c in claims if c["claim_id"] == claim_id)
    payout, _ = payout_calculation(policy_id, **args)
    expected = labels[claim_id]["payout"]
    print(f"{claim_id}: calculator ${payout:,.2f}, handler ${expected:,.2f} {'OK' if abs(payout - expected) <= 1 else 'MISMATCH'}")

## Part 1: one change at a time

The versions build on each other and all run on the same model, so the differences come from the changes and not from the model. Luna at `low` is the reference because the lab gives its numbers. Change `ABLATION_MODEL` to repeat this on another model.

Two of the changes are tested two ways, because it isn't obvious which is better:

- **History:** the manual says to check the history on *every* claim (section 4, step 9). By the design principle, that makes it a fixed step for code, not a tool. But a tool lets the agent skip it when it can't matter. Both are tested.
- **Manual:** a short summary in the prompt costs tokens on every call. A lookup tool costs only when the agent uses it, but adds a round trip, and a small model may not use it when it should.

In [ ]:
ABLATION_MODEL, ABLATION_EFFORT = "openai/gpt-6-luna", "low"
RUNS = 3

BASE = dict(policy="tool", end_on_record=False, calculator=False, rules=False, history=None, manual=None, wording="full")

versions = {
    "0 lab agent":            BASE,
    "1 fixed steps in code":  {**BASE, "policy": "inject", "end_on_record": True},
    "2 + calculator":         {**BASE, "policy": "inject", "end_on_record": True, "calculator": True},
    "3 + rules":              {**BASE, "policy": "inject", "end_on_record": True, "calculator": True, "rules": True},
    "4a + history injected":  {**BASE, "policy": "inject", "end_on_record": True, "calculator": True, "rules": True, "history": "inject"},
    "4b + history tool":      {**BASE, "policy": "inject", "end_on_record": True, "calculator": True, "rules": True, "history": "tool"},
    "5a + manual summary":    {**BASE, "policy": "inject", "end_on_record": True, "calculator": True, "rules": True, "history": "inject", "manual": "summary"},
    "5b + manual tool":       {**BASE, "policy": "inject", "end_on_record": True, "calculator": True, "rules": True, "history": "inject", "manual": "tool"},
}

# What each version sends: prompt sizes and the tools the agent can call
preview = []
for name, cfg in versions.items():
    tools = [t.name for t in make_tools(cfg, claims[0])]
    preview.append({"version": name, "system prompt words": len(make_system_prompt(cfg).split()),
                    "user prompt words": len(make_user_prompt(claims[0], cfg).split()), "tools": ", ".join(tools)})
pd.DataFrame(preview)

In [ ]:
ablation = []
for name, cfg in versions.items():
    ablation += run_version(name, cfg, ABLATION_MODEL, ABLATION_EFFORT, claims, labels, runs=RUNS)

In [ ]:
summarize(ablation)

### Results from my run (2026-11-16)

| Version | Worst | Mean | Best | Cost per run | Seconds per run | Calls per claim |
|---|---|---|---|---|---|---|
| 0 lab agent | 17 | 17.7 | 18 | $0.0073 | 42 | 3.0 |
| 1 fixed steps in code | 12 | 13.3 | 15 | $0.0031 | 337* | 1.0 |
| 2 + calculator | 17 | 17.7 | 19 | $0.0044 | 21 | 1.5 |
| 3 + rules | 18 | 18.7 | 19 | $0.0046 | 20 | 1.6 |
| 4a + history injected | 15 | 17.7 | 19 | $0.0046 | 19 | 1.6 |
| 4b + history tool | 18 | 19.7 | 22 | $0.0060 | 32 | 2.3 |
| 5a + manual summary | 20 | 20.7 | 22 | $0.0047 | 24 | 1.5 |
| 5b + manual tool | 21 | 21.7 | 22 | $0.0098 | 36 | 3.3 |

\* Most likely a slow or retrying provider: version 1 had the fewest calls and the lowest cost.

The next three cells break the results down: which claims each version got right (out of 3 runs), the kinds of mistakes, and the score at each difficulty.

In [ ]:
# Each number = how many of the 3 runs got this claim right. Claims every version always got right are hidden.
df = pd.DataFrame(ablation)
matrix = df.pivot_table(index=["claim_id", "difficulty", "handler"], columns="version", values="right", aggfunc="sum", sort=False)
runs = df["run"].nunique()
matrix[(matrix < runs).any(axis=1)]

In [ ]:
def error_type(r):
    if r["error"]:
        return "error"
    if r["decision"] == "nothing recorded":
        return "no decision recorded"
    if r["decision"] == r["handler"]:
        return "right decision, wrong payout"
    return f"wrong {r['decision']}"

misses = df[~df["right"]].copy()
misses["error_type"] = misses.apply(error_type, axis=1)
pd.crosstab(misses["version"], misses["error_type"])

In [ ]:
by_difficulty(ablation)

**Mean right at each difficulty, from my run.** The queue has 15 routine claims, 5 hard ones (payout arithmetic and incomplete carrier letters) and 5 that need the manual or the customer's history.

| Version | Routine /15 | Hard /5 | Manual /5 |
|---|---|---|---|
| 0 lab agent | 14.0 | 3.7 | 0.0 |
| 1 fixed steps | 11.3 | 1.0 | 1.0 |
| 2 + calculator | 13.3 | 3.7 | 0.7 |
| 3 + rules | 13.0 | 5.0 | 0.7 |
| 4a history injected | 12.7 | 4.7 | 0.3 |
| 4b history tool | 13.7 | 5.0 | 1.0 |
| 5a manual summary | 11.3 | 5.0 | 4.3 |
| 5b manual tool | 12.3 | 4.7 | 4.7 |

**Mistake types for four of the versions (total over 3 runs).** A wrong approval pays money that isn't owed, so it's the costliest mistake.

| Version | Right decision, wrong payout | Wrong approve | Wrong decline | Wrong refer |
|---|---|---|---|---|
| 0 lab agent | 5 | 9 | 0 | 8 |
| 2 + calculator | 0 | 9 | 3 | 10 |
| 5a manual summary | 3 | 2 | 1 | 7 |
| 5b manual tool | 1 | 6 | 0 | 3 |

### What each change did

- **1, fixed steps in code: worse (17 to 12).** Removing the `get_policy` round trip also removed the model's thinking time: it now had to commit in its first reply, and `record_decision` asks for the decision before the rationale. A round trip that looks wasteful can be doing useful work.
- **2, calculator: wrong payouts from 5 to 0**, and the three arithmetic claims (CLM-1011, 1014, 1021) right. Working out the payout also gave the model its reasoning step back.
- **3, rules: fixed both incomplete carrier letters** (CLM-1023 and 1025, 3 of 3). The grouping rules are company knowledge that no model can infer, not a crutch for small models.
- **4a against 4b, history: the tool did better** (worst 18 against 15). Injected history with no rule on what it means seems to have distracted the model. It worked once 5a added that rule.
- **5a against 5b, manual: the summary took the manual claims from 0 to 4.3 of 5.** The tool scored 1 more but cost twice as much (3.3 calls a claim). The summary also cut wrong approvals from 9 to 2. But 5a lost routine claims (14.0 to 11.3), which Part 2 explains.

## Part 2: error analysis and version 6

5a is the base for the next version rather than 5b: its worst run is only 1 lower, at half the cost and with a third of the wrong approvals. The cells below list its misses and show the rationales.

In [ ]:
CHECK = "5a + manual summary"   # the version to examine
best = misses[misses["version"] == CHECK]
summary = best.groupby("claim_id").agg(
    difficulty=("difficulty", "first"), missed_runs=("run", "count"),
    handler=("handler", "first"), handler_payout=("handler_payout", "first"),
    model_said=("decision", lambda d: ", ".join(sorted(set(d)))),
    payouts=("payout", lambda p: ", ".join(f"${x:,.0f}" for x in sorted(set(p)))),
    error_types=("error_type", lambda e: ", ".join(sorted(set(e)))),
).reset_index()
summary["what the claim tests"] = summary["claim_id"].map(lambda c: labels[c]["tests"])
summary

In [ ]:
show_misses(ablation, "5a + manual summary", ABLATION_MODEL, run=1)

**What the misses showed (my run).** Most of the routine claims 5a lost come from the summary itself: it added reasons to refer without the manual's counterweights.

- **CLM-1001:** the summary said meals can be paid at $35 a day without receipts, but left out "if there are receipts, pay the receipts". The model replaced $95 of receipted meals with $35 or $70.
- **CLM-1002:** it left out "items of $100 or less need only the customer's list". "Four shirts, $120" was read as one item over $100 and referred.
- **CLM-1022:** "a missing document means refer" clashed with the meal allowance.
- **CLM-1010 and 1012:** over-referral, because the summary didn't say "refer only for these reasons".
- **CLM-1013:** missed the waiting period (Economy needs 12 hours; the delay was 7). It had failed since version 3.

### Version 6: the fixes

Built on 5a. Each fix comes from the claims manual or the plan data, not from the handlers' answers:

1. **A completed manual summary** (`MANUAL_SUMMARY_V2`). It adds the counterweights: refer only for the listed reasons, proof of value only for single items above $100, and pay meal receipts when there are any, with the flat allowance only when there are none.
2. **The plan limits injected by code** (`plan_limits_text`), with a line to check the benefit's conditions, such as the waiting period, against them first.
3. **Rationale before decision** in `record_decision`, so the model reasons before it commits.

The second run below is the same version with the short policy wording (about 800 words instead of 1,400), to see whether the tokens saved cost any accuracy.

In [ ]:
versions["6 + fixes"] = {**versions["5a + manual summary"], "manual": "summary_v2", "plan_limits": True, "rationale_first": True}
versions["6 + fixes, short wording"] = {**versions["6 + fixes"], "wording": "short"}

for name in ["6 + fixes", "6 + fixes, short wording"]:
    ablation += run_version(name, versions[name], ABLATION_MODEL, ABLATION_EFFORT, claims, labels, runs=RUNS)
summarize(ablation)

**Results from my run (2026-11-16).** Version 6 ran in a later session than 0 to 5b, on the same model and settings.

| Version | Worst | Mean | Best | Cost per run | Seconds per run | Calls per claim |
|---|---|---|---|---|---|---|
| 5a + manual summary | 20 | 20.7 | 22 | $0.0047 | 24 | 1.5 |
| 6 + fixes | 22 | 23.0 | 24 | $0.0047 | 19 | 1.6 |
| 6 + fixes, short wording | 22 | 23.3 | 25 | $0.0047 | 22 | 1.5 |

Version 6 gains 2 on the worst run at the same cost as 5a. The three fixes went in together, so this doesn't show how much each one is worth.

The short wording kept the same worst run and had one perfect run, so it's the version for the held-out check. It looks barely cheaper because of prompt caching: each version's first run pays full price (about $0.0069), and later runs cost about half once the system prompt is cached. With a warm cache, the short wording saves about 12%.

## Part 3: the held-out check

Version 6 was built by looking at the queue's mistakes, so its queue score flatters it. The 25 held-out claims have the same mix (15 routine, 5 hard, 5 manual) and were never looked at while tuning. The version was fixed before this run and nothing changed afterwards.

In [ ]:
BEST = "6 + fixes, short wording"
held = run_version(BEST, versions[BEST], ABLATION_MODEL, ABLATION_EFFORT, held_claims, held_labels, runs=3, dataset="held out")
held_lab = run_version("0 lab agent", versions["0 lab agent"], ABLATION_MODEL, ABLATION_EFFORT, held_claims, held_labels, runs=3, dataset="held out")
summarize(held + held_lab)

In [ ]:
by_difficulty(held + held_lab)

In [ ]:
held_df = pd.DataFrame(held + held_lab)
held_misses = held_df[~held_df["right"]].copy()
held_misses["error_type"] = held_misses.apply(error_type, axis=1)
pd.crosstab(held_misses["version"], held_misses["error_type"])

**Results from my run (2026-11-18).**

| Agent | Runs | Worst | Mean | Cost per run | Seconds per run | Calls per claim |
|---|---|---|---|---|---|---|
| 6 + fixes, short wording | 24, 23, 22 | 22 | 23.0 | $0.0049 | 24 | 1.6 |
| 0 lab agent | 17, 17, 17 | 17 | 17.0 | $0.0070 | 36 | 3.0 |

| Agent | Routine /15 | Hard /5 | Manual /5 |
|---|---|---|---|
| 6 + fixes, short wording | 14.0 | 5.0 | 4.0 |
| 0 lab agent | 14.3 | 2.7 | 0.0 |

- **The improvement holds on unseen claims.** Both agents score almost exactly what they scored on the queue (22 and 23.3 on the queue for version 6; 17 and 17.7 for the lab's agent).
- **The whole gain is on the hard and manual claims.** On routine claims the two are level. The lab's agent gets none of the manual claims on either set, because the rules it needs are only in the manual.
- **Version 6 is cheaper and faster as well.** The first run cost $0.0068 and the next two about $0.0040, which is the caching effect again.

## Accuracy against cost

Each point is one version on one set of claims, at its worst run. Up and to the left is better. The dashed line is the target of 15.

In [ ]:
everything = summarize(ablation + held + held_lab, by=("version", "model", "effort", "dataset"))
fig = px.scatter(everything, x="cost_per_run", y="worst", color="version", symbol="dataset",
                 hover_data=["mean", "seconds_per_run", "calls_per_claim"],
                 title="Worst run against cost per run of 25 claims (Luna, low)")
fig.add_hline(y=15, line_dash="dash")
fig.show()

## Findings

- **Move fixed steps into code, but measure it.** It cut the calls from 3 a claim to 1 and the cost by more than half, but on its own it lost 5 claims on the worst run, because the round trip it removed had been the model's thinking time. The calculator and putting the rationale first gave that back.
- **The model for judgement, code for arithmetic.** The calculator took wrong payouts from 5 to 0 at almost no cost.
- **Missing information is not a model problem.** The lab's agent got none of the manual claims on either set. A bigger model can't know rules it never sees, but a short summary in the prompt can tell it.
- **A summary needs its counterweights.** The first summary listed reasons to refer and left out the limits on them, so the model over-referred routine claims. Completing it from the manual fixed that.
- **Prompt or tool depends on the cost of a round trip.** For the manual, the summary scored about as well as the tool at half the cost. For the history, injecting it only worked once the prompt said what to do with it.
- **Judge on the worst run, and check on held-out data.** The requirement is "every run", and the queue was used for tuning. Version 6's held-out score matches its queue score, so the gain isn't an artefact of tuning.

**Limits and next steps**

- Everything ran on one model, Luna at `low`. I haven't run the search for the cheapest model that keeps version 6 at 15 or more on every run. With a worst run of 22 there is room for a cheaper model, and changing `ABLATION_MODEL` is enough to test one.
- 25 claims and 3 runs each are enough to trust a gap of 5 claims, not to rank versions a claim apart.
- Version 6's three fixes went in together, so their separate value isn't measured.